# 📘 LLM Zoomcamp — Homework 01: Agentic RAG

**Student:** Mohammed Sheikh-Noor  
**Course:** LLM Zoomcamp (DataTalksClub)  
**Repo:** https://github.com/mohanoor-ai/LLMZoomCamp  
**Date:** June 2026


In [130]:
import os

In [131]:
from dotenv import load_dotenv

In [132]:
from minsearch import Index

In [133]:
from gitsource import GithubRepositoryDataReader

In [164]:
import toyaikit

In [165]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [166]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

MODEL = "openai/gpt-oss-20b:free"

In [167]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
Q1. How many lesson pages
How many lesson pages are in the dataset?

24
**72**
240
720

In [77]:
documents = [f.parse() for f in files]
len(documents)

72

In [ ]:
2. Indexing and searching (1 point)
01-agentic-rag/lessons/03-rag.md
**01-agentic-rag/lessons/14-agentic-loop.md**
04-evaluation/lessons/13-llm-as-judge.md
06-best-practices/lessons/02-hybrid-search.md

In [78]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [79]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query)

results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [80]:
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [ ]:
3. RAG (1 point)
700
**7000**
70000
700000

In [81]:
def search(query):
    return index.search(query)


def build_context(query):
    results = search(query)

    context = ""

    for doc in results[:3]:
        context += f"Filename: {doc['filename']}\n"
        context += f"Content:\n{doc['content']}\n\n"

    return context

In [82]:
def llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response

In [83]:
def rag(query):
    context = build_context(query)

    prompt = f"""
You're a course teaching assistant.

Answer the QUESTION using only the CONTEXT.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm(prompt)

    answer = response.choices[0].message.content

    return answer, response.usage

In [23]:
question = "How does the agentic loop keep calling the model until it stops?"

answer, usage = rag(question)

print(answer)

The agentic loop keeps calling the model by wrapping the request‑response cycle in a `while True` loop.  
At each iteration it:

1. Sends the current conversation (`messages`) to the model, including any tools that are available.  
2. Gets back a response that may contain **function calls** or a final **message**.  
3. If the response contains a function call, the loop executes that function, appends the tool’s output back to `messages`, and sets a flag (`has_function_calls = True`).  
4. If the response contains only a message (no function calls), `has_function_calls` remains `False`.

After processing the response, the loop checks the flag:

```python
if has_function_calls == False:
    break
```

When the model’s last turn contains no function calls, the flag is `False` and the loop exits. Thus, the loop continues to request new model turns as long as the model keeps asking for tools, automatically ending when the model provides a final answer with no more tool requests.


In [24]:
usage.prompt_tokens

5756

In [ ]:
4. Chunking (1 point)
70
**295**
1100
4500

In [25]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [26]:
from gitsource import chunk_documents

In [27]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [28]:
len(chunks)

295

In [29]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [ ]:
5. RAG with chunking (1 point)
about the same
**3x fewer**
10x fewer
30x fewer

In [30]:
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [31]:
def chunk_search(query):
    return chunk_index.search(query)

In [32]:
def build_chunk_context(query):
    results = chunk_search(query)

    context = ""

    for doc in results[:3]:
        context += f"Filename: {doc['filename']}\n"
        context += f"Content:\n{doc['content']}\n\n"

    return context

In [33]:
question = "How does the agentic loop keep calling the model until it stops?"

context = build_chunk_context(question)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{question}
"""

response = llm(prompt)

In [34]:
response.usage.prompt_tokens

1516

In [ ]:
5756 / response.usage.prompt_tokens

In [ ]:
6. Turning it into an agent (1 point)
0
**4**
10
20

In [168]:
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")

In [169]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"]
)

In [170]:
llm_client = OpenAIClient(
    model="openai/gpt-oss-20b:free",
    client=client
)

In [171]:
search_calls = 0

def search(query: str) -> str:
    """
    Search the course lesson chunks and return relevant context.
    """
    global search_calls
    search_calls += 1

    results = chunk_index.search(query)
    return "\n\n".join(r["content"] for r in results[:3])

In [172]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [173]:
developer_prompt = """
You are a teaching assistant.

IMPORTANT RULES:
- You MUST call the search tool at least 4 times before answering.
- You MUST vary your queries each time (do not repeat similar searches).
- Do not answer until you have made at least 4 searches.

This is mandatory.
"""

In [174]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [175]:
search_calls = 0

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

-> Response received


-> Response received


-> Response received


-> Response received


In [179]:
search_calls

3